<a href="https://colab.research.google.com/github/SebaAcunaC/YOLO11-Vigilancia-Urbana-USACH/blob/main/CodigoFinal_YOLO11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# PROYECTO DE FIN DE GRADO - INGENIERÍA USACH
# Master Notebook: Benchmarking Multi-Modelo y Reconocimiento Facial
# ==============================================================================

# 1. SETUP DE REPRODUCIBILIDAD Y LIMPIEZA
!pip install ultralytics roboflow torchvision -q

import os, yaml, torch, time, gc
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO
from roboflow import Roboflow
from torchvision.models.detection import ssd300_vgg16, fasterrcnn_resnet50_fpn

# Configuración de Hardware
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Trabajando con: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

def limpiar_memoria():
    torch.cuda.empty_cache()
    gc.collect()

# 2. DESCARGA DINÁMICA DE DATASET (Semana 2)
api_key = input("Introduce tu API Key de Roboflow: ")
rf = Roboflow(api_key=api_key)
project = rf.workspace("sebastianacua").project("person-hgivm-vyoba")
dataset = project.version(1).download("yolov8")

# 3. MÓDULO DE RECONOCIMIENTO FACIAL (Requerimiento Profesor)
print("\n--- Iniciando Módulo de Rostros ---")
# Usamos el modelo pre-entrenado de rostros para comparar latencia
model_face = YOLO("yolo11n-face.pt")
res_face = model_face.predict(source=dataset.location + "/valid/images", imgsz=640, stream=True)
for r in res_face: last_face = r
t_face = sum(last_face.speed.values())
print(f"Latencia Rostros: {t_face:.2f} ms | FPS: {1000/t_face:.2f}")

limpiar_memoria()

# 4. BENCHMARKING MAESTRO (YOLO11 vs YOLOv8 vs Legacy)
# Aplicamos el barrido de resoluciones de la Semana 5
modelos_nombres = ["yolo11n.pt", "yolov8n.pt"]
resoluciones = [1088, 640, 320]
resultados_bench = []

for m_name in modelos_nombres:
    model = YOLO(m_name)
    for res in resoluciones:
        print(f"Probando {m_name} a {res}p...")
        results = model.predict(source=dataset.location + "/valid/images", imgsz=res, stream=True, verbose=False)
        for r in results: last_r = r
        t_total = sum(last_r.speed.values())
        resultados_bench.append({'Modelo': m_name, 'Res': res, 'FPS': 1000/t_total})
    del model
    limpiar_memoria()

# 5. COMPARATIVA CONTRA MODELOS LEGACY (SSD / Faster R-CNN)
# Este bloque justifica tu elección ante la conferencia
print("\n--- Evaluando Modelos Legacy (SSD/R-CNN) ---")
model_rcnn = fasterrcnn_resnet50_fpn(pretrained=True).to(device).eval()
# Simulamos una inferencia simple para obtener tiempo de respuesta
dummy_img = torch.randn(1, 3, 640, 640).to(device)
start = time.time()
with torch.no_grad(): model_rcnn(dummy_img)
t_rcnn = (time.time() - start) * 1000
print(f"Faster R-CNN Latencia: {t_rcnn:.2f} ms | FPS: {1000/t_rcnn:.2f}")

del model_rcnn
limpiar_memoria()

# 6. GENERACIÓN DE TABLA DE RESULTADOS FINAL
df_master = pd.DataFrame(resultados_bench)
print("\n=== RESUMEN DE INGENIERÍA FINAL ===")
print(df_master.groupby(['Modelo', 'Res'])['FPS'].mean().unstack())